<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c03-header.png" alt="Nextia Learning · Essential Mathematics and Statistics for AI" width="100%">

# Solution: Vector operations

**[Vector operations](https://learning.nextia-ai.com/courses/math/m02/vector-operations/)** · Essential Mathematics and Statistics for AI · Module 2, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**Optional: check the numbers in Python.** This notebook is optional. The lesson works without code: a calculator is enough. Use the notebook to check the lesson's numbers and to redo its worked examples and tasks in Python.

**You will** add and scale vectors, calculate Priya's urgency scores with the dot product, measure the length of a vector, check the shapes of a matrix product, and read and fix Tomás's shape error.

| | |
|---|---|
| **Time** | About 25 minutes, after you read the lesson |
| **Needs** | numpy 2.5.3, which the first setup cell installs, for the matrix sections only. No account. It runs in Colab, Kaggle or any Jupyter. |
| **You should know** | How a ticket becomes a feature vector, and Priya's matrix of T-4101 to T-4103, from [Represent observations: Worked example: Priya builds the feature matrix](https://learning.nextia-ai.com/courses/math/m02/represent-observations/#worked-example-priya-builds-the-feature-matrix). The vectors of T-4104 and T-4105 from [Represent observations: Guided practice: add two tickets](https://learning.nextia-ai.com/courses/math/m02/represent-observations/#guided-practice-add-two-tickets). |
| **Tested** | Python 3.14.6 with numpy 2.5.3, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

The [lesson page](https://learning.nextia-ai.com/courses/math/m02/vector-operations/) has the full explanations, the charts, the explorers and the knowledge checks that count toward your certificate. Read a section there, then check its numbers here.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, replace each `...` with your calculation, run the cell, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C03/M02-L02-vector-operations/vector-operations-solution.ipynb).

## Setup: install the package

Run the next cell. It installs `numpy` 2.5.3, the version that the course uses. It needs an internet connection and takes up to a minute the first time. Colab and Kaggle already have numpy, so there it is usually quick.

You should see `numpy 2.5.3`. If you see another version, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q numpy==2.5.3
import numpy; print("numpy", numpy.__version__)

## Setup: the helpers

The next cell defines three small functions that the notebook uses. You do not need to read them, but you can.

- `expect(what, yours, expected)` compares your number with the lesson's number. It says "Check passed." or if yours is too high or too low. It does not show the answer.
- `table(headers, rows)` prints a small table with aligned columns.
- `bars(labels, counts)` prints a text bar chart: one row of `█` for each label.

Run the cell. You should see `Ready.`

In [ ]:
# Helpers: check an answer without showing it, print a table, draw text bars.
import math, random, statistics

def expect(what, yours, expected, tol=0.01):
    """Compare your value with the expected one, to within `tol` for numbers."""
    if yours is ...:
        print(f"Not yet: replace ... with your calculation for {what}, then run the cell again.")
        return
    if isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        if abs(yours - expected) <= tol:
            print(f"Check passed: {what} is {yours:g}.")
        else:
            side = "too high" if yours > expected else "too low"
            print(f"Not yet: your {what}, {yours:g}, is {side}. Look at the step before it again.")
    elif yours == expected:
        print(f"Check passed: {what} is {yours}.")
    else:
        print(f"Not yet: your {what} is {yours!r}. That is not the expected answer.")

def table(headers, rows, digits=2):
    """Print rows as a table. Floats are rounded to `digits` decimals."""
    cell = lambda v: f"{v:,.{digits}f}" if isinstance(v, float) else str(v)
    text = [[cell(v) for v in row] for row in rows]
    widths = [max(len(str(h)), *(len(r[i]) for r in text)) for i, h in enumerate(headers)]
    print("  ".join(str(h).rjust(w) for h, w in zip(headers, widths)))
    for r in text:
        print("  ".join(v.rjust(w) for v, w in zip(r, widths)))

def bars(labels, counts, width=40):
    """Print a text bar chart: the longest bar is `width` characters."""
    top = max(counts) or 1
    size = max(len(str(label)) for label in labels)
    for label, n in zip(labels, counts):
        print(f"{str(label).rjust(size)} | {'█' * round(n / top * width)} {n}")

print("Ready.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Add and scale vectors

Two tickets from the same customer, with two counts each: refund words and delivery words. **a** = (3, 1) and **b** = (1, 2). Addition goes position by position. **Scaling** multiplies every position by one number.

The next cell writes both operations as small functions with plain Python lists.

> **Predict.** What are a + b, b + a, 2 × a and 0.5 × a? Then run the cell.

In [ ]:
def add(u, v):
    """Add two vectors position by position."""
    return [x + y for x, y in zip(u, v)]

def scale(c, v):
    """Multiply every position of v by the number c."""
    return [c * x for x in v]

a = [3, 1]
b = [1, 2]
print("a + b  ", add(a, b))
print("b + a  ", add(b, a))
print("2 × a  ", scale(2, a))
print("0.5 × a", scale(0.5, a))

> **Check.** You should see `[4, 3]` twice: the order of addition does not change the sum. Then `[6, 2]` and `[1.5, 0.5]`: the same direction, a different length.

The **mean of vectors** uses both operations: add, then scale by 1 ÷ (how many there are). Run the cell. You should see `[2.0, 1.5]`.

In [ ]:
print("mean of a and b:", scale(1 / 2, add(a, b)))

## 2. The dot product: a weighted score

Priya's weights **w** = (0.5, 2, 1, 0.5) are in the feature order of Lesson 1: length, refund words, past tickets, photo. The **dot product** multiplies position by position, then adds.

Run the cell. You should see the products `1.0`, `4`, `0` and `0.5`, and the score `5.5`.

In [ ]:
features = ["length", "refund words", "past tickets", "photo"]
w = [0.5, 2, 1, 0.5]
t4101 = [2, 2, 0, 1]

def dot(u, v):
    """Multiply position by position, then add the products."""
    return sum(x * y for x, y in zip(u, v))

table(["Feature", "Value", "Weight", "Value × weight"],
      [[f, x, wi, x * wi] for f, x, wi in zip(features, t4101, w)], digits=1)
print("score w · x =", dot(w, t4101))

Now all three tickets of Lesson 1.

> **Predict.** Which ticket gets the highest score? Then run the cell.

In [ ]:
X = {
    "T-4101": [2, 2, 0, 1],
    "T-4102": [1, 0, 3, 0],
    "T-4103": [3, 1, 1, 1],
}
table(["Ticket", "Products with w", "Score w · x"],
      [[t, " + ".join(f"{x * wi:g}" for x, wi in zip(v, w)), float(dot(w, v))] for t, v in X.items()], digits=1)

> **Check.** You should see the scores 5.5, 3.5 and 5.0, as in the lesson. T-4101 is first. The score is not a probability: it only puts the tickets in order.

### What the dot product measures

On the lesson page, the [explorer](https://learning.nextia-ai.com/courses/math/m02/vector-operations/#what-the-dot-product-measures) lets you drag b. Here you do the same with a list. The **projection** of b on a is the part of b that points along a. Its length is a · b ÷ (the length of a).

> **Predict.** a = (3, 1). What is the sign of a · b for b = (1, 2), b = (−1, 3) and b = (−2, 1)? Then run the cell.

In [ ]:
for b_try in [[1, 2], [-1, 3], [-2, 1]]:
    d = dot(a, b_try)
    projection = d / math.sqrt(dot(a, a))
    print(f"b = {str(b_try):8}  a · b = {d:3}   projection length {projection:6.3f}")

> **Check.** You should see 5, 0 and −5. The projection lengths are 1.581, 0.000 and −1.581. The dot product is positive for a similar direction, 0 at a right angle, and negative for opposite directions. And 3.162 × 1.581 = 5: the length of a times the projection.

## 3. The length of a vector

The **length** (or **norm**) is the square root of the sum of the squares. It is also the square root of a · a.

Run the cell. You should see 3.162 for a and 3.0 for T-4101.

In [ ]:
def length(v):
    """The length of v: the square root of v · v."""
    return math.sqrt(dot(v, v))

print("a · a      ", dot(a, a))
print("length of a", round(length(a), 3))
print("length of T-4101", length(t4101))

A **unit vector** has length 1. Divide each number by the length.

> **Predict.** What is a as a unit vector? Then run the cell.

In [ ]:
unit_a = scale(1 / length(a), a)
print("unit vector:", [round(x, 3) for x in unit_a])
print("its length: ", round(length(unit_a), 3))

> **Check.** You should see `[0.949, 0.316]` and a length of `1.0`. It points the same way as a.

## 4. Matrix multiplication: many dot products at once

Priya scores the whole matrix X. She writes w as a column: a matrix with shape 4 × 1. Each row of X "dot" the column gives one score. A short function does it with lists: `zip(*B)` gives the columns of B.

Run the cell. You should see `[[5.5], [3.5], [5.0]]`: the same three scores.

In [ ]:
def matmul(A, B):
    """Every row of A with every column of B: one dot product for each pair."""
    return [[dot(row, column) for column in zip(*B)] for row in A]

X_rows = list(X.values())                 # shape 3 × 4
w_column = [[0.5], [2], [1], [0.5]]       # shape 4 × 1
print(matmul(X_rows, w_column))

The shape rule: **(n × k) × (k × m) → (n × m)**. The two inner numbers must be equal. The two outer numbers give the shape of the result. The next cell writes the rule as a function and checks the rows of the lesson's table.

> **Predict.** Which row does not fit? Then run the cell.

In [ ]:
def product_shape(first, second):
    """The shape of first × second, or None if the inner numbers differ."""
    return (first[0], second[1]) if first[1] == second[0] else None

pairs = [((3, 4), (4, 1)), ((1200, 4), (4, 1)), ((1200, 4), (4, 5)),
         ((1200, 4), (5, 4)), ((32, 784), (784, 10)), ((1, 384), (384, 1200))]
table(["First", "Second", "Result"], [[f, s, product_shape(f, s) or "does not fit"] for f, s in pairs])

> **Check.** You should see `does not fit` only for `(1200, 4)` and `(5, 4)`: 4 ≠ 5. The last row, `(1, 1200)`, is the start of a search: 1,200 dot products in one step.

Now the same with numpy, the library that a professional uses. In numpy, `@` is matrix multiplication and `.shape` gives the shape. Run the cell. You should see `[5.5 3.5 5. ]` and `(3,)`.

In [ ]:
import numpy as np

X_np = np.array(X_rows)          # shape (3, 4)
w_np = np.array(w)               # a vector of 4 numbers
scores = X_np @ w_np
print(scores)
print(scores.shape)

numpy prints `5.` for 5.0. The shape `(3,)` means a vector of 3 numbers: numpy does not need w as a 4 × 1 column.

## 5. Worked example: Tomás's shape error

Tomás has a weight matrix W for the 5 teams: one row for each team, one column for each feature. **The lesson does not give these weights, so the next cell uses made-up ones.** It also needs a week of 1,200 tickets: here, Priya's three tickets repeated 400 times.

> **Predict.** What error message do you get for the product of X (1,200 × 4) and W (5 × 4)? Then run the cell.

In [ ]:
teams = ["returns", "delivery", "payment", "warranty", "account"]
W = np.array([          # made-up weights: one row for each team
    [0.5, 2, 0, 1],     # returns
    [0.5, 0, 1, 0],     # delivery
    [0.5, 3, 0, 0],     # payment
    [1,   0, 0, 2],     # warranty
    [0,   0, 2, 0],     # account
])
X_week = np.tile(X_np, (400, 1))   # a stand-in week: 1,200 rows
print("X_week", X_week.shape, " W", W.shape)

try:
    np.dot(X_week, W)
except ValueError as error:
    print("ValueError:", error)

> **Check.** You should see `ValueError: shapes (1200,4) and (5,4) not aligned: 4 (dim 1) != 5 (dim 0)`: the message from the lesson. (With `@`, numpy writes a longer message that says the same: size 5 is different from 4.)

Priya's fix is the **transpose**: rows become columns. Then she checks the result shape and one value by hand: T-4101 (row 1 of X) and the returns team (column 1 of the transposed W).

Run the cell. You should see `W.T (4, 5)`, `result (1200, 5)`, and `6.0` twice.

In [ ]:
print("W.T", W.T.shape)
result = X_week @ W.T
print("result", result.shape)

by_hand = dot([2, 2, 0, 1], [0.5, 2, 0, 1])   # T-4101 · the returns weights
print("cell (1, 1) by hand:", by_hand)
print("cell (1, 1) by numpy:", result[0, 0])

## 6. Guided practice: score two tickets

Use Priya's weights w = (0.5, 2, 1, 0.5) and the two tickets from Lesson 1: T-4104 = (4, 2, 2, 0) and T-4105 = (1, 0, 0, 0). The functions `dot` and `length` are defined above.

> **Your turn.** Replace each `...`. Calculate both scores, put all five tickets in order from the highest score to the lowest, calculate the length of T-4104, and give the shape of the result X × w when X has 5 rows. Then run the check cell.

In [ ]:
t4104 = [4, 2, 2, 0]
t4105 = [1, 0, 0, 0]

score_4104 = dot(w, t4104)
score_4105 = dot(w, t4105)
all_tickets = {**X, "T-4104": t4104, "T-4105": t4105}
order = sorted(all_tickets, key=lambda t: dot(w, all_tickets[t]), reverse=True)
length_4104 = length(t4104)
shape_result = product_shape((5, 4), (4, 1))

Run the check cell.

In [ ]:
expect("the score of T-4104", score_4104, 8)
expect("the score of T-4105", score_4105, 0.5)
expect("the order", order, ["T-4104", "T-4101", "T-4103", "T-4102", "T-4105"])
expect("the length of T-4104", length_4104, 4.899, tol=0.001)
expect("the shape of the result", shape_result, (5, 1))

T-4105 asks to change an address, so a very low score is what Grace expects. If your score for T-4105 is high, check that the weights are in the same order as the features.

## 7. Independent variation: change one weight

Grace says that a customer who writes again and again is the most urgent case. Priya changes one weight: past tickets from 1 to 3.

> **Your turn.** Write the new weights. Calculate the new scores of T-4101, T-4102 and T-4103, and their new order, highest first. Then run the check cell.

In [ ]:
w_new = [0.5, 2, 3, 0.5]
new_4101 = dot(w_new, X["T-4101"])
new_4102 = dot(w_new, X["T-4102"])
new_4103 = dot(w_new, X["T-4103"])
new_order = sorted(X, key=lambda t: dot(w_new, X[t]), reverse=True)

Run the check cell.

In [ ]:
expect("the new score of T-4101", new_4101, 5.5)
expect("the new score of T-4102", new_4102, 9.5)
expect("the new score of T-4103", new_4103, 7.0)
expect("the new order", new_order, ["T-4102", "T-4103", "T-4101"])

T-4102 moves from last to first: it has the most past tickets (3), and that feature now counts three times as much. T-4101 does not change, because it has 0 past tickets.

## 8. Failure case: making the shapes fit with the wrong operation

Tomás changes W from 5 × 4 to 4 × 5 with `reshape` instead of the transpose. The error goes away.

> **Predict.** Is the returns column of `W.reshape(4, 5)` the same as the returns column of `W.T`? Then run the cell.

In [ ]:
print("W.T, column 1 (returns):         ", W.T[:, 0])
print("W.reshape(4, 5), column 1:        ", W.reshape(4, 5)[:, 0])

wrong = X_week @ W.reshape(4, 5)
print("shape with reshape:", wrong.shape, " no error")
print("cell (1, 1) with reshape:  ", wrong[0, 0])
print("cell (1, 1) with transpose:", result[0, 0])

> **Check.** You should see the returns weights `[0.5 2.  0.  1. ]` from the transpose, but `[0.5 0.  0.  2. ]` from the reshape. The reshape reads the 20 numbers row by row and refills them, so the teams' weights get mixed. The shape `(1200, 5)` fits and no error appears, but cell (1, 1) is 3.0 instead of 6.0.

**Fix:** say what each direction means ("rows are teams, columns are features"), use the transpose, and check one cell by hand, as in section 5.

## Check your understanding and recap

Answer the **knowledge checks** at the end of the [lesson page](https://learning.nextia-ai.com/courses/math/m02/vector-operations/). They count toward your certificate when you are signed in and enrolled. The notebook does not have them.

In this notebook you added and scaled vectors, and calculated Priya's urgency scores with the dot product: 5.5, 3.5 and 5.0. You measured lengths (√10 ≈ 3.162 and 3), made a unit vector, and checked the shape rule (n × k) × (k × m) → (n × m) with plain Python and with numpy. You read Tomás's shape error, fixed it with a transpose, and saw that a reshape fits the shapes but mixes the weights.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Similarity and distance](https://learning.nextia-ai.com/courses/math/m02/similarity-and-distance/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/math/m02/vector-operations/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield tickets are fictional and have no real people.